# Logistic Regression con validacion LOSO

Este notebook implementa el **Experimento A**: features derivadas de HR/R-R y ECG.

El objetivo es evaluar la generalizacion a un trabajador no visto durante el entrenamiento mediante Leave-One-Subject-Out Cross-Validation (LOSO). El target se crea dentro de cada fold usando unicamente `FatigueIndex` del conjunto TRAIN.

## Regla de no leakage

En cada fold se siguen estos pasos:

1. Un trabajador queda como TEST y los otros cuatro forman TRAIN.
2. Se calculan P33 y P66 usando solo `FatigueIndex` de TRAIN.
3. Se crean las etiquetas `Low`, `Medium` y `High` con esos percentiles.
4. Se aplica exactamente la misma categorizacion al TEST.
5. Cualquier imputador o scaler se ajusta solo con TRAIN.

Las entradas son las columnas `_z`, ya normalizadas respecto a la baseline individual de cada trabajador. No se usa `FatigueIndex` como feature.

In [7]:
from pathlib import Path

import json
import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'nootebooks' else Path.cwd()
DATA_DIR = ROOT_DIR
BASELINE_WINDOW_COUNT = 15
TARGET_COLUMN_CANDIDATES = ['FatigueIndex', 'fatigue_index']
LABELS = ['Low', 'Medium', 'High']

print(f'Directorio de datos: {DATA_DIR}')

Directorio de datos: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\new


## Carga de datos

Cada CSV corresponde a un trabajador. El nombre de la carpeta (`W00`, ..., `W04`) se conserva como identificador para construir los folds, pero nunca entra como feature del modelo.

In [8]:
worker_files = {
    worker_dir.name: worker_dir / 'PROCESSED' / 'combined_features_1min.csv'
    for worker_dir in sorted(DATA_DIR.glob('W*'))
    if (worker_dir / 'PROCESSED' / 'combined_features_1min.csv').exists()
}
if len(worker_files) != 5:
    raise ValueError(f'Se esperaban 5 trabajadores y se encontraron {len(worker_files)}: {list(worker_files)}')

data = {}
for worker, path in worker_files.items():
    frame = pd.read_csv(path)
    target_matches = [column for column in TARGET_COLUMN_CANDIDATES if column in frame.columns]
    if len(target_matches) != 1:
        raise ValueError(f'{worker}: no se encontro exactamente una columna FatigueIndex/fatigue_index')
    frame = frame.rename(columns={target_matches[0]: 'FatigueIndex'})
    frame['Trabajador'] = worker
    data[worker] = frame

feature_columns = [
    'ECG_energy_z',
    'ECG_mean_z',
    'ECG_missing_peaks_z',
    'ECG_range_z',
    'ECG_samp_ent_z',
    'ECG_std_z',
    'HR_max_z',
    'HR_mean_z',
    'HR_min_z',
    'RMSSD_z',
    'SDNN_z',
    'pNN50_z',
]
missing_features = [
    column for column in feature_columns
    if column not in data[next(iter(data))].columns
]
if missing_features:
    raise ValueError(f'Faltan features requeridas: {missing_features}')
if not feature_columns:
    raise ValueError('No se encontraron columnas de entrada terminadas en _z')

print('Trabajadores:', list(data))
print('Features del Experimento A:', feature_columns)
print('Ventanas:', {worker: len(frame) for worker, frame in data.items()})

Trabajadores: ['W00', 'W01', 'W02', 'W03', 'W04']
Features del Experimento A: ['ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z', 'ECG_samp_ent_z', 'ECG_std_z', 'HR_max_z', 'HR_mean_z', 'HR_min_z', 'RMSSD_z', 'SDNN_z', 'pNN50_z']
Ventanas: {'W00': 104, 'W01': 637, 'W02': 603, 'W03': 407, 'W04': 562}


## Funciones de target y evaluacion

Los percentiles se calculan en cada fold y solo con el vector de `FatigueIndex` de TRAIN. Los valores faltantes del target no se usan para calcular percentiles ni para entrenar/evaluar ese fold.

In [9]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(
        fatigue_values,
        bins=[-np.inf, p33, p66, np.inf],
        labels=LABELS,
        right=False,
    ).astype(object)


def get_train_percentiles(train_frame):
    train_fatigue = pd.to_numeric(train_frame['FatigueIndex'], errors='coerce').dropna()
    if train_fatigue.empty:
        raise ValueError('TRAIN no contiene valores validos de FatigueIndex')
    return np.percentile(train_fatigue, [33, 66])


def build_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=2000, random_state=42)),
    ])

## Experimento A

Features utilizadas: todas las columnas `_z` disponibles, que corresponden a features derivadas de HR/R-R y ECG.

La matriz de confusion utiliza siempre el orden `Low`, `Medium`, `High`.

In [10]:
fold_results = []
confusion_matrices = {}

for test_worker in sorted(data):
    train_workers = [worker for worker in sorted(data) if worker != test_worker]
    train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
    test_frame = data[test_worker].copy()

    p33, p66 = get_train_percentiles(train_frame)
    train_target = make_labels(
        pd.to_numeric(train_frame['FatigueIndex'], errors='coerce'), p33, p66
    )
    test_target = make_labels(
        pd.to_numeric(test_frame['FatigueIndex'], errors='coerce'), p33, p66
    )

    train_mask = train_target.notna()
    test_mask = test_target.notna()
    X_train = train_frame.loc[train_mask, feature_columns]
    y_train = train_target.loc[train_mask]
    X_test = test_frame.loc[test_mask, feature_columns]
    y_test = test_target.loc[test_mask]

    if y_train.nunique() < 2:
        raise ValueError(f'{test_worker}: TRAIN tiene menos de dos clases')
    if y_test.empty:
        raise ValueError(f'{test_worker}: TEST no contiene FatigueIndex valido')

    model = build_model()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    confusion_matrices[test_worker] = confusion_matrix(
        y_test, y_pred, labels=LABELS
    )
    matrix = confusion_matrices[test_worker]
    class_recall = np.diag(matrix) / matrix.sum(axis=1)
    fold_results.append({
        'Test_worker': test_worker,
        'Train_workers': ', '.join(train_workers),
        'P33_train': p33,
        'P66_train': p66,
        'N_train': len(y_train),
        'N_test': len(y_test),
        'Recall_Low': class_recall[0],
        'Recall_Medium': class_recall[1],
        'Recall_High': class_recall[2],
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0),
    })

fold_results_df = pd.DataFrame(fold_results)
mean_row = {column: np.nan for column in fold_results_df.columns}
mean_row['Test_worker'] = 'Media'
mean_row['Train_workers'] = 'Promedio de los folds'
for metric in ['P33_train', 'P66_train', 'N_train', 'N_test', 'Recall_Low', 'Recall_Medium', 'Recall_High', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']:
    mean_row[metric] = fold_results_df[metric].mean()
fold_results_with_mean = pd.concat(
    [fold_results_df, pd.DataFrame([mean_row])], ignore_index=True
)
fold_results_with_mean

,Test_worker,Train_workers,P33_train,P66_train,N_train,N_test,Recall_Low,Recall_Medium,Recall_High,Accuracy,Balanced_Accuracy,Macro_F1
0,W00,"W01, W02, W03, W04",-0.197880,1.820980,2209.0,104.0,0.952381,0.750000,0.250000,0.855769,0.650794,0.602767
1,W01,"W00, W02, W03, W04",-0.355762,1.500986,1676.0,637.0,0.766355,0.368030,0.984674,0.687598,0.706353,0.698997
2,W02,"W00, W01, W03, W04",-0.367063,1.608196,1710.0,603.0,0.764045,0.826990,0.497778,0.694859,0.696271,0.691612
3,W03,"W00, W01, W02, W04",-0.323504,0.923629,1906.0,407.0,0.969231,0.480000,0.886435,0.874693,0.778555,0.717805
4,W04,"W00, W01, W02, W03",0.207435,3.015125,1751.0,562.0,0.818898,0.595745,0.285714,0.793594,0.566786,0.551766
5,Media,Promedio de los folds,-0.207355,1.773783,1850.4,462.6,0.854182,0.604153,0.580920,0.781303,0.679752,0.652589


In [11]:
print('Matrices de confusion por trabajador TEST:')
for worker, matrix in confusion_matrices.items():
    print(f'\nTEST = {worker}')
    print(pd.DataFrame(matrix, index=LABELS, columns=LABELS))

metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1']
summary_df = pd.DataFrame({
    'Metric': metrics,
    'Mean': [fold_results_df[metric].mean() for metric in metrics],
    'Std': [fold_results_df[metric].std(ddof=1) for metric in metrics],
})
print('Media y desviacion estandar de las metricas:')
display(summary_df)

aggregate_confusion = np.sum(
    np.stack([confusion_matrices[worker] for worker in sorted(confusion_matrices)]),
    axis=0,
)
print('Matriz de confusion agregada de los cinco folds:')
display(pd.DataFrame(aggregate_confusion, index=LABELS, columns=LABELS))

class_support = aggregate_confusion.sum(axis=1)
predicted_support = aggregate_confusion.sum(axis=0)
class_metrics_df = pd.DataFrame({
    'Support': class_support,
    'Recall': np.diag(aggregate_confusion) / class_support,
    'Precision': np.diag(aggregate_confusion) / predicted_support,
}, index=LABELS)
print('Metricas agregadas por clase:')
display(class_metrics_df.round(3))

distribution_rows = []
for result in fold_results:
    worker = result['Test_worker']
    fatigue_values = pd.to_numeric(data[worker]['FatigueIndex'], errors='coerce')
    labels = make_labels(fatigue_values, result['P33_train'], result['P66_train'])
    counts = labels.value_counts().reindex(LABELS, fill_value=0)
    distribution_rows.append({
        'Test_worker': worker,
        'Low': int(counts['Low']),
        'Medium': int(counts['Medium']),
        'High': int(counts['High']),
        'Total_valid': int(counts.sum()),
    })
print('Distribucion de clases en cada trabajador TEST:')
display(pd.DataFrame(distribution_rows))

Matrices de confusion por trabajador TEST:

TEST = W00
        Low  Medium  High
Low      80       4     0
Medium    2       6     0
High      0       9     3

TEST = W01
        Low  Medium  High
Low      82      23     2
Medium    5      99   165
High      0       4   257

TEST = W02
        Low  Medium  High
Low      68      18     3
Medium   30     239    20
High      0     113   112

TEST = W03
        Low  Medium  High
Low      63       2     0
Medium   13      12     0
High      0      36   281

TEST = W04
        Low  Medium  High
Low     416      92     0
Medium   19      28     0
High      3       2     2
Media y desviacion estandar de las metricas:


,Metric,Mean,Std
0,Accuracy,0.781303,0.087566
1,Balanced_Accuracy,0.679752,0.078011
2,Macro_F1,0.652589,0.071724


Matriz de confusion agregada de los cinco folds:


,Low,Medium,High
Low,709,139,5
Medium,69,384,185
High,3,164,655


Metricas agregadas por clase:


,Support,Recall,Precision
Low,853,0.831,0.908
Medium,638,0.602,0.559
High,822,0.797,0.775


Distribucion de clases en cada trabajador TEST:


,Test_worker,Low,Medium,High,Total_valid
0,W00,84,8,12,104
1,W01,107,269,261,637
2,W02,89,289,225,603
3,W03,65,25,317,407
4,W04,508,47,7,562


## Experimento B: solo features HR/R-R

Pendiente. Se conservara el mismo protocolo LOSO, la misma creacion del target dentro de cada fold y el mismo tratamiento de imputacion/scaling. Solo cambiara la lista de columnas de entrada.

## Experimento C: solo features ECG

Pendiente. Se conservara el mismo protocolo LOSO, la misma creacion del target dentro de cada fold y el mismo tratamiento de imputacion/scaling. Solo cambiara la lista de columnas de entrada.

## Modelo final y exportacion

La evaluacion LOSO anterior mide la generalizacion dejando un trabajador fuera en cada fold. Despues, este bloque reentrena un modelo final con todos los trabajadores y todas las ventanas validas. Este modelo final es el que se utilizara para la aplicacion.

El modelo y el preprocesamiento completo se guardan en formato `joblib`, el modelo para Expo/React Native en formato `ONNX`, y la configuracion en un archivo `JSON`. En este notebook se guardan en `new/models/original/`.

In [12]:
# Reentrenamiento final con todos los trabajadores y exportacion de artefactos
all_frame = pd.concat([data[worker] for worker in sorted(data)], ignore_index=True)
all_fatigue = pd.to_numeric(all_frame['FatigueIndex'], errors='coerce')
final_p33, final_p66 = np.percentile(all_fatigue.dropna(), [33, 66])
all_target = make_labels(all_fatigue, final_p33, final_p66)
all_mask = all_target.notna()
X_all = all_frame.loc[all_mask, feature_columns]
y_all = all_target.loc[all_mask]

final_model = build_model()
final_model.fit(X_all, y_all)

models_dir = ROOT_DIR / 'models' / 'original'
models_dir.mkdir(parents=True, exist_ok=True)
joblib_path = models_dir / 'final_logistic_regression.joblib'
onnx_path = models_dir / 'final_logistic_regression.onnx'
metadata_path = models_dir / 'final_logistic_regression_metadata.json'

joblib.dump(final_model, joblib_path)
onnx_model = convert_sklearn(
    final_model,
    initial_types=[('features', FloatTensorType([None, len(feature_columns)]))],
)
onnx_path.write_bytes(onnx_model.SerializeToString())

metadata = {
    'model_type': 'LogisticRegression',
    'validation': 'LOSO',
    'training_workers': sorted(data),
    'feature_columns': feature_columns,
    'feature_count': len(feature_columns),
    'labels': LABELS,
    'p33_train_all_workers': float(final_p33),
    'p66_train_all_workers': float(final_p66),
    'target_column': 'FatigueIndex',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Modelo final entrenado con {len(y_all)} ventanas y {len(feature_columns)} features')
print(f'Guardado en: {models_dir}')

Modelo final entrenado con 2313 ventanas y 12 features
Guardado en: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\new\models\original
